# 04 – Fine-tune Cellpose-SAM on Fluo-N2DL-HeLa (Kaggle, GPU)

**Purpose:** Fine-tune the pretrained Cellpose-SAM model on this dataset and test whether it segments
better than the pretrained model.

**Runs on:** Kaggle Notebook with GPU (not locally). Input: the Fluo-N2DL-HeLa dataset attached via *Add Input*.

### Design
- **Train on sequence 02**, silver-truth labels (complete coverage, all 92 frames).
  Every 10th frame is held out to monitor the test loss during training.
- **Evaluate on sequence 01**, gold truth – never seen during training. Same metrics as notebook 03.
- Gold truth is not used for training: it is only partially annotated, so unannotated nuclei would be learned as background.
- Limitation: the model learns the outline style of the silver truth, not of the manual annotation.

**Training parameters:** defaults of the official Cellpose-SAM training notebook
(100 epochs, learning rate 1e-5, weight decay 0.1, batch size 1).

## 1. Check GPU and install Cellpose
The Cellpose version is pinned to the one used locally (4.2.1.1) so that results are comparable.
Requires *Settings → Accelerator → GPU* and *Settings → Internet → on* (for pip).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q cellpose==4.2.1.1 tifffile

## 2. Imports and settings
The dataset folder is searched under `/kaggle/input`, because its exact mount path depends on how it was added.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage.segmentation import relabel_sequential
from cellpose import models, core, train, io

io.logger_setup()   # prints training progress

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")   # everything saved here appears in the notebook output

DATA = next(p for p in INPUT.rglob("Fluo-N2DL-HeLa") if (p / "01").is_dir())
SEQ_TRAIN, SEQ_EVAL = "02", "01"
TEST_EVERY = 10
FULL_FRAMES = [13, 52]
IOU_THRESHOLD = 0.5

N_EPOCHS, LEARNING_RATE, WEIGHT_DECAY, BATCH_SIZE = 100, 1e-5, 0.1, 1
MODEL_NAME = "cpsam_hela_seq02"

print("data:", DATA)
print("GPU available:", core.use_gpu())

## 3. Load training data (sequence 02, silver truth)
Image `tTTT.tif` belongs to mask `man_segTTT.tif`. Images are passed as 2D arrays;
Cellpose converts them to its 3-channel input and normalises them internally.

In [ ]:
train_x, train_y, test_x, test_y = [], [], [], []
for img_path in sorted((DATA / SEQ_TRAIN).glob("t*.tif")):
    t = int(img_path.stem[1:])
    mask_path = DATA / f"{SEQ_TRAIN}_ST" / "SEG" / f"man_seg{t:03d}.tif"
    if not mask_path.exists():
        continue
    img, mask = tifffile.imread(img_path), tifffile.imread(mask_path)
    if t % TEST_EVERY == 0:
        test_x.append(img); test_y.append(mask)
    else:
        train_x.append(img); train_y.append(mask)

print(f"train: {len(train_x)} | test: {len(test_x)} | image shape {train_x[0].shape}, {train_x[0].dtype}")

## 4. Fine-tune
Starts from the pretrained Cellpose-SAM weights. Progress (loss per epoch) is printed by Cellpose.
The model is saved to `/kaggle/working/models/`.

In [ ]:
model = models.CellposeModel(gpu=True)

t0 = time.time()
model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=train_x, train_labels=train_y,
    test_data=test_x, test_labels=test_y,
    batch_size=BATCH_SIZE, n_epochs=N_EPOCHS,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    nimg_per_epoch=len(train_x),
    save_path=WORK, model_name=MODEL_NAME,
)
train_minutes = (time.time() - t0) / 60
print(f"training time: {train_minutes:.0f} min | model: {model_path}")

## 5. Learning curves
Training loss should fall; the test loss (held-out frames of sequence 02) shows whether the model
generalises or starts to overfit (test loss rising again). Test loss is only computed at some epochs.

In [ ]:
epochs = np.arange(1, len(train_losses) + 1)
has_test = np.asarray(test_losses) > 0

plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, label="train")
plt.plot(epochs[has_test], np.asarray(test_losses)[has_test], "o-", label="test (held-out seq 02)")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.tight_layout()
plt.savefig(WORK / "learning_curves.png", dpi=150)
plt.show()

pd.DataFrame({"epoch": epochs, "train_loss": train_losses, "test_loss": test_losses}) \
  .to_csv(WORK / "learning_curves.csv", index=False)

## 6. Evaluation functions
Identical to notebook 03 (IoU > 0.5 matching; detection metrics on fully annotated frames;
SEG-like score per reference nucleus on all gold-truth frames).

In [ ]:
def iou_matrix(gt_lab, pred_lab):
    n_gt, n_pred = int(gt_lab.max()), int(pred_lab.max())
    codes = gt_lab.ravel().astype(np.int64) * (n_pred + 1) + pred_lab.ravel().astype(np.int64)
    overlap = np.bincount(codes, minlength=(n_gt + 1) * (n_pred + 1)).reshape(n_gt + 1, n_pred + 1)
    union = overlap.sum(axis=1, keepdims=True) + overlap.sum(axis=0, keepdims=True) - overlap
    return (overlap / np.maximum(union, 1))[1:, 1:]


def evaluate(gt_lab, pred_lab, thresh=IOU_THRESHOLD):
    gt_lab = relabel_sequential(gt_lab)[0]
    pred_lab = relabel_sequential(pred_lab)[0]
    iou = iou_matrix(gt_lab, pred_lab)
    n_gt, n_pred = iou.shape
    hits = iou > thresh
    tp = int(hits.sum())
    best = iou.max(axis=1) if n_pred else np.zeros(n_gt)
    return {
        "n_gt": n_gt, "n_pred": n_pred, "tp": tp, "fp": n_pred - tp, "fn": n_gt - tp,
        "precision": tp / n_pred if n_pred else np.nan,
        "recall": tp / n_gt if n_gt else np.nan,
        "f1": 2 * tp / (n_gt + n_pred) if (n_gt + n_pred) else np.nan,
        "mean_iou_matched": iou[hits].mean() if tp else np.nan,
        "seg_like": np.where(best > thresh, best, 0).mean() if n_gt else np.nan,
    }

## 7. Evaluate pretrained vs. fine-tuned on sequence 01
Both models segment the same 28 gold-truth frames of the unseen sequence.

In [ ]:
gt_files = sorted((DATA / f"{SEQ_EVAL}_GT" / "SEG").glob("man_seg*.tif"))
frames = [int("".join(c for c in f.stem if c.isdigit())) for f in gt_files]
gt = {t: tifffile.imread(f) for t, f in zip(frames, gt_files)}
raw = {t: tifffile.imread(DATA / SEQ_EVAL / f"t{t:03d}.tif") for t in frames}

candidates = {
    "pretrained": models.CellposeModel(gpu=True),
    "fine-tuned": models.CellposeModel(gpu=True, pretrained_model=str(model_path)),
}
pred = {}
for name, m in candidates.items():
    t0 = time.time()
    pred[name] = {t: m.eval(raw[t])[0] for t in frames}
    print(f"{name}: {len(frames)} frames in {time.time() - t0:.0f} s")

In [ ]:
rows = [{"frame": t, "method": name, "full_frame": t in FULL_FRAMES, **evaluate(gt[t], pred[name][t])}
        for name in pred for t in frames]
res = pd.DataFrame(rows)
res.to_csv(WORK / "metrics_pretrained_vs_finetuned_seq01.csv", index=False)

cols = ["frame", "method", "n_gt", "n_pred", "tp", "fp", "fn", "precision", "recall", "f1", "mean_iou_matched"]
print("Fully annotated frames:")
print(res[res["full_frame"]][cols].round(3).to_string(index=False))

weighted = res.groupby("method").apply(
    lambda d: np.average(d["seg_like"], weights=d["n_gt"]), include_groups=False)
print("\nSEG-like score, all GT frames, weighted by annotated nuclei:")
print(weighted.round(3).to_string())

partial = res[~res["full_frame"]].pivot(index="frame", columns="method", values="seg_like")
print("\nSEG-like score on partial frames (selected, often difficult nuclei):")
print(partial.round(3).to_string())

## Results
*(to be filled in after running)*

- Training time / GPU:
- Learning curves:
- Fully annotated frames (13, 52), pretrained vs. fine-tuned:
- SEG-like score (weighted):
- Partial frames (difficult nuclei, e.g. mitoses):